In [1]:
import os
import re
import pandas as pd
from tqdm import tqdm
import numpy as np
import psycopg2
import matplotlib.pyplot as plt
import seaborn as sns
import nest_asyncio

In [2]:
icustays = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/icu/icustays.csv.gz')
admissions = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/hosp/admissions.csv.gz')
pts = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/hosp/patients.csv.gz')
pts['yob']= pts['anchor_year'] - pts['anchor_age']  # get yob to ensure a given visit is from an adult
pts['min_valid_year'] = pts['anchor_year'] + (2022 - pts['anchor_year_group'].str.slice(start=-4).astype(int))
pts = pd.merge(icustays,pts,on='subject_id',how='left')
pts = pd.merge(pts,admissions[['hadm_id', 'admittime', 'dischtime', 'deathtime',
       'admission_type', 'admit_provider_id', 'admission_location',
       'discharge_location', 'insurance', 'language', 'marital_status', 'race',
       'edregtime', 'edouttime', 'hospital_expire_flag']],on='hadm_id',how='left')
pts.shape

(94458, 29)

In [3]:
pts['admittime'] = pd.to_datetime(pts['admittime'])
pts['Age']=pts['admittime'].dt.year - pts['yob']
pts = pts.loc[pts['Age'] >= 18]
pts.shape

(94458, 30)

In [4]:
pts = pts[pts.los>=1]
pts.shape

(74829, 30)

In [5]:
pts[['Age','los']].describe()

,Age,los
count,74829.000000,74829.000000
mean,65.085274,4.400381
std,16.365507,5.828026
min,18.000000,1.000000
25%,55.000000,1.599919
50%,67.000000,2.504618
75%,77.000000,4.702303
max,103.000000,226.403079


In [39]:
pts.shape

(74829, 30)

In [41]:
pts.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit',
       'intime', 'outtime', 'los', 'gender', 'anchor_age', 'anchor_year',
       'anchor_year_group', 'dod', 'yob', 'min_valid_year', 'admittime',
       'dischtime', 'deathtime', 'admission_type', 'admit_provider_id',
       'admission_location', 'discharge_location', 'insurance', 'language',
       'marital_status', 'race', 'edregtime', 'edouttime',
       'hospital_expire_flag', 'Age'],
      dtype='object')

In [42]:
pts['intime'] = pd.to_datetime(pts['intime'])
pts['outtime'] = pd.to_datetime(pts['outtime'])

# 标记第一次住院（first_hosp_stay）
pts['first_hosp_stay'] = pts.groupby('subject_id')['admittime'].transform('min') == pts['admittime']
pts['first_hosp_stay'] = pts['first_hosp_stay'].astype(int)

# 标记每次住院的第一次 ICU（first_icu_stay）
pts['first_icu_stay'] = pts.groupby(['subject_id', 'hadm_id'])['intime'].transform('min') == pts['intime']
pts['first_icu_stay'] = pts['first_icu_stay'].astype(int)

# 新增 'admin_counts' 列: 统计每个病人的住院次数
pts['admin_counts'] = pts.groupby('subject_id')['admittime'].rank(method='first').astype(int)

# 新增 'icu_counts' 列: 统计每个住院的 ICU 次数
pts['icu_counts'] = pts.groupby(['hadm_id'])['intime'].rank(method='first').astype(int)

In [44]:
missing_cols = pts.columns[pts.isnull().any()]
missing_cols

Index(['dod', 'deathtime', 'admit_provider_id', 'discharge_location',
       'insurance', 'language', 'marital_status', 'edregtime', 'edouttime'],
      dtype='object')

In [45]:
pts.to_csv('D:/2024BI_Journal/MIMIC_IV/Median_B/All_democ.csv',index=False)

In [7]:
df = pts.copy()
print(len(df.subject_id.unique()),len(df.hadm_id.unique()),len(df.stay_id.unique()))

54551 68546 74829


In [8]:
microbiologyevents = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/hosp/microbiologyevents.csv.gz')
microevents = microbiologyevents[microbiologyevents['hadm_id'].isin(df['hadm_id'])]
del(microbiologyevents)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_3988\3507416237.py:1: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  microbiologyevents = pd.read_csv('D:/mimic-iv-3.1/mimiciv/3.1/hosp/microbiologyevents.csv.gz')


In [9]:
microevents = microevents[['microevent_id','micro_specimen_id', 'subject_id', 'hadm_id',
                           'chartdate', 'charttime', 'spec_itemid',
       'spec_type_desc', 'storedate', 'storetime', 
       'test_name', 'org_itemid', 'org_name','ab_name', 
       'interpretation']]
#细菌感染检查
microevents.shape

(776142, 15)

In [10]:
merged = pd.merge(microevents, df[['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime',
                                   'admittime','los']], on =['subject_id', 'hadm_id'], how='left')

merged['intime'] = pd.to_datetime(merged['intime'])
merged['outtime'] = pd.to_datetime(merged['outtime'])
merged['admittime'] = pd.to_datetime(merged['admittime'])

merged['charttime'] = pd.to_datetime(merged['charttime'])
merged['storetime'] = pd.to_datetime(merged['storetime'])

# 在ICU_admit之前发生MDR感染

In [11]:
# 细菌培养在ICU前
SSC_icu_window_criteria = merged['charttime'].between(merged['admittime'],merged['intime'])

merged = merged[SSC_icu_window_criteria]
merged.shape

(244767, 20)

In [12]:
merged = merged.sort_values(['stay_id','intime','charttime','org_name']).drop_duplicates(['stay_id','charttime','test_name','org_name','ab_name', 'interpretation']) 

In [13]:
BI = merged.dropna(subset = ['org_name'])
BI.shape

(87295, 20)

In [14]:
org_name_sel = BI[['org_itemid', 'org_name']]
org_name_sel = org_name_sel.drop_duplicates(subset=['org_name'], keep='first')
org_name_sel.shape

(305, 2)

In [15]:
# 分类函数
def classify_bacteria(org_name):
    org_name = org_name.upper()  # 转换为大写，便于匹配
    
    # 分类规则
    if "STAPH AUREUS" in org_name:
        return "Staphylococcus aureus"
    elif any(keyword in org_name for keyword in [
        "ENTEROBACTER", "ESCHERICHIA", "KLEBSIELLA", 
        "CITROBACTER", "MORGANELLA", "PROVIDENCIA", 
        "SALMONELLA"]):
        return "Enterobacteriaceae"
    elif "ENTEROCOCCUS" in org_name:
        return "Enterococcus spp."
    elif "PSEUDOMONAS AERUGINOSA" in org_name:
        return "Pseudomonas aeruginosa"
    elif "ACINETOBACTER" in org_name:
        return "Acinetobacter spp."
    else:
        return "Others"

In [16]:
org_name_sel['org_category'] = org_name_sel['org_name'].apply(classify_bacteria)

In [17]:
org_name_sel[org_name_sel.org_category=='Staphylococcus aureus']

,org_itemid,org_name,org_category
434675,80023.0,STAPH AUREUS COAG +,Staphylococcus aureus
545836,80293.0,POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS,Staphylococcus aureus


In [18]:
print(BI.shape,len(pd.unique(BI.subject_id)),len(pd.unique(BI.hadm_id)),len(pd.unique(BI.stay_id)))

(87295, 20) 6332 6723 7901


In [19]:
MDR_CAL_cha = BI[['stay_id','micro_specimen_id','org_name','ab_name','interpretation']].set_index(['stay_id','micro_specimen_id','org_name','ab_name'], append=True).unstack(-1).reset_index()
MDR_CAL_cha = MDR_CAL_cha.iloc[:,1:]
MDR_CAL_cha.columns = ['stay_id','micro_specimen_id','org_name'] + MDR_CAL_cha['interpretation'].columns.to_list()
MDR_CAL_cha.drop(columns = np.nan, inplace = True)
MDR_CAL_cha.shape

(87295, 45)

In [20]:
#细菌培养的抗生素的名称列表
MDR_CAL_cha_test = MDR_CAL_cha.copy()
ab_list = MDR_CAL_cha.columns.to_list()
ab_list[0:3]

['stay_id', 'micro_specimen_id', 'org_name']

In [21]:
del ab_list[0:3]
for i in ab_list:
    MDR_CAL_cha_test[i] = MDR_CAL_cha_test[i].astype(str)

MDR_CAL_cha_test = MDR_CAL_cha_test.groupby(['stay_id','micro_specimen_id','org_name']).sum().reset_index()

In [22]:
def find_r(x):
    if re.search('R',x):
        return 'R'
    elif re.search('I',x):
        return 'I'
    elif re.search('S',x):
        return 'S'
    else:
        return np.nan

for i in ab_list:
    MDR_CAL_cha_test[i] = MDR_CAL_cha_test[i].apply(lambda x:find_r(x))

MDR_CAL_conc = MDR_CAL_cha_test.drop_duplicates()
MDR_CAL_conc.shape

(22182, 45)

In [23]:
MDR_CAL_cat = pd.merge(MDR_CAL_conc,org_name_sel[['org_name','org_category']],\
how = 'left', on = ['org_name'])#.org_category.value_counts()
MDR_CAL_cat.org_category.value_counts()

Others                    11526
Staphylococcus aureus      3889
Enterobacteriaceae         3288
Enterococcus spp.          2402
Pseudomonas aeruginosa      951
Acinetobacter spp.          126
Name: org_category, dtype: int64

In [24]:
#5个不同组别的微生物的名称
org_pieces = dict(list(org_name_sel['org_name'].groupby(org_name_sel['org_category'])))

St_name = org_pieces['Staphylococcus aureus'].tolist()
Eb_name = org_pieces['Enterobacteriaceae'].tolist()
Ec_name = org_pieces['Enterococcus spp.'].tolist()
Ps_name = org_pieces['Pseudomonas aeruginosa'].tolist()
Ac_name = org_pieces['Acinetobacter spp.'].tolist()

Eb_exc_amino_list = ['PROVIDENCIA RETTGERI', 'PROVIDENCIA STUARTII']
Eb_exc_cefazolin_list = [
    'CITROBACTER FREUNDII COMPLEX', 'ENTEROBACTER AEROGENES',
    'ENTEROBACTER CLOACAE COMPLEX', 'ENTEROBACTER CLOACAE', 'HAFNIA ALVEI',
    'MORGANELLA MORGANII', 'PROTEUS PENNERI', 'PROTEUS VULGARIS',
    'PROTEUS VULGARIS GROUP', 'PROVIDENCIA RETTGERI', 'PROVIDENCIA STUARTII',
    'SERRATIA MARCESCENS'
]
Eb_exc_cefuroxime_list = [
    'MORGANELLA MORGANII', 'PROTEUS PENNERI', 'PROTEUS VULGARIS',
    'PROTEUS VULGARIS GROUP', 'SERRATIA MARCESCENS'
]
Eb_exc_ampicillin_list = [
    'CITROBACTER KOSERI', 'CITROBACTER FREUNDII COMPLEX',
    'ENTEROBACTER AEROGENES', 'ENTEROBACTER CLOACAE COMPLEX',
    'ENTEROBACTER CLOACAE', 'HAFNIA ALVEI', 'KLEBSIELLA PNEUMONIAE',
    'KLEBSIELLA OXYTOCA', 'MORGANELLA MORGANII', 'PROTEUS PENNERI',
    'PROTEUS VULGARIS', 'PROTEUS VULGARIS GROUP', 'PROVIDENCIA RETTGERI',
    'PROVIDENCIA STUARTII', 'SERRATIA MARCESCENS'
]
Eb_exc_ampicillinsulbactam_list = [
    'CITROBACTER FREUNDII COMPLEX', 'CITROBACTER KOSERI',
    'ENTEROBACTER AEROGENES', 'ENTEROBACTER CLOACAE COMPLEX',
    'ENTEROBACTER CLOACAE', 'HAFNIA ALVEI', 'PROVIDENCIA RETTGERI',
    'SERRATIA MARCESCENS'
]
Eb_exc_teracycline_list = [
    'MORGANELLA MORGANII', 'PROTEUS MIRABILIS','PROTEUS PENNERI',
    'PROTEUS VULGARIS', 'PROTEUS VULGARIS GROUP', 'PROVIDENCIA RETTGERI',
    'PROVIDENCIA STUARTII'
]

In [25]:
def MDR(df):
    #金黄色葡萄球菌
    if df['org_name'] in St_name:
        if df['org_name'] == 'POSITIVE FOR METHICILLIN RESISTANT STAPH AUREUS':
            return 'MRSA', 1
        elif df['org_name'] == 'S. AUREUS POSITIVE; MRSA POSITIVE':
            return 'MRSA', 1
        else:
            St_dict = {}
            St_sum = 0
            St_ab_list = []
            #氨基糖苷-庆大霉素
            if df['GENTAMICIN'] == 'I' or df['GENTAMICIN'] == 'R':
                St_dict['Aminoglycosides'] = 1
            #安沙霉素-利福平
            if df['RIFAMPIN'] == 'I' or df['RIFAMPIN'] == 'R':
                St_dict['Ansamycins'] = 1
            #抗MRSA 头孢菌素类
            #抗葡萄球菌β内酰胺或头孢
            if (df[['AMPICILLIN','AMPICILLIN/SULBACTAM','CEFAZOLIN',\
                'CEFEPIME','CEFTAZIDIME','CEFTRIAXONE','CEFUROXIME',\
                'OXACILLIN','PENICILLIN G','PIPERACILLIN','PIPERACILLIN/TAZO']] == 'I').any() or \
                (df[['AMPICILLIN','AMPICILLIN/SULBACTAM','CEFAZOLIN',\
                'CEFEPIME','CEFTAZIDIME','CEFTRIAXONE','CEFUROXIME',\
                'OXACILLIN','PENICILLIN G','PIPERACILLIN','PIPERACILLIN/TAZO']] == 'R').any():
                St_dict['LactamsCephamycins'] = 1
            #氟喹诺酮-环丙沙星
            if df['CIPROFLOXACIN']=='I' or df['CIPROFLOXACIN']=='R':
                St_dict['Fluoroquinolones'] = 1
            #叶酸途径抑制剂-甲氧苄啶 复方新诺明
            if df['TRIMETHOPRIM/SULFA']=='I' or df['TRIMETHOPRIM/SULFA']=='R':
                St_dict['Folate'] = 1
            #Fucidanes
            #糖肽类-万古霉素
            if df['VANCOMYCIN'] == 'I' or df['VANCOMYCIN'] == 'R':
                St_dict['Glycopeptides'] = 1
            #甘氨环素
            #林可酰胺类-克林霉素
            if df['CLINDAMYCIN'] == 'I' or df['CLINDAMYCIN'] == 'R':
                St_dict['Lincosamides'] = 1
            #脂肽类-达托霉素
            if df['DAPTOMYCIN'] == 'I' or df['DAPTOMYCIN'] == 'R':
                St_dict['Lipopeptides'] = 1
            #大环内酯类-红霉素
            if df['ERYTHROMYCIN'] == 'I' or df['ERYTHROMYCIN'] == 'R':
                St_dict['Macrolides'] = 1
            #恶唑烷酮类-利奈唑胺
            if df['LINEZOLID'] == 'I' or df['LINEZOLID'] == 'R':
                St_dict['Oxazolidinones'] = 1
            #苯丙醇类
            #磷酶酸类
            #链阳霉素类
            #四环素类-四环素
            if df['TETRACYCLINE'] == 'I' or df['TETRACYCLINE'] == 'R':
                St_dict['Tetracyclines'] = 1
            
            for i in St_dict:
                St_sum += St_dict[i]
                St_ab_list.append(i)
            if St_sum>=3:
                return ";".join(St_ab_list), 1
            else:
                return ";".join(St_ab_list), 0
    #肠球菌
    elif df['org_name'] in Ec_name:
        Ec_dict = {}
        Ec_sum = 0
        Ec_ab_list = []
        #氨基糖苷-高浓度庆大霉素
        #链霉素-链霉素
        #碳青霉烯类-亚胺培南、美罗培南
        if (df[['IMIPENEM','MEROPENEM']] == 'I').any() or \
                (df[['IMIPENEM','MEROPENEM']] == 'R').any():
            if df['org_name'] != 'ENTEROCOCCUS FAECIUM': 
#只有不是屎肠球菌才执行这条命令。如果是屎肠球菌，则跳过，字典里不会纳入碳青霉烯类的信息。
                Ec_dict['Carbapenems'] = 1
        #氟喹诺酮-环丙沙星、左氧氟沙星
        if (df[['CIPROFLOXACIN','LEVOFLOXACIN']]=='I').any() or \
                (df[['CIPROFLOXACIN','LEVOFLOXACIN']]=='R').any():
            Ec_dict['Fluroquinolones'] = 1
        #糖肽类-万古霉素
        if df['VANCOMYCIN'] == 'I' or df['VANCOMYCIN'] == 'R':
            Ec_dict['Glycopeptides'] = 1
        #甘氨环素类
        #脂肽类-达托霉素
        if df['DAPTOMYCIN'] == 'I' or df['DAPTOMYCIN'] == 'R':
            Ec_dict['Lipopeptides'] = 1
        #恶唑烷酮类-利奈唑胺
        if df['LINEZOLID'] == 'I' or df['LINEZOLID'] == 'R':
            Ec_dict['Oxazolidinones'] = 1
        #青霉素类-氨苄西林
        if df['AMPICILLIN'] == 'I' or df['AMPICILLIN'] == 'R':
            Ec_dict['Penicillins'] = 1
        #链阳霉素类
        #四环素类
        if df['TETRACYCLINE'] == 'I' or df['TETRACYCLINE'] == 'R':
            Ec_dict['Tetracyclines'] = 1
        
        for i in Ec_dict:
            Ec_sum += Ec_dict[i]
            Ec_ab_list.append(i)
        if Ec_sum>=3:
            return ";".join(Ec_ab_list), 1
        else:
            return ";".join(Ec_ab_list), 0
    #肠杆菌
    elif df['org_name'] in Eb_name:
        Eb_dict = {}
        Eb_sum = 0
        Eb_ab_list = []
        #氨基糖苷-庆大霉素、妥布霉素、阿米卡星
        if (df[['GENTAMICIN','TOBRAMYCIN']] == 'I').any() \
            or (df[['GENTAMICIN','TOBRAMYCIN']] == 'R').any():
            if df['org_name'] not in Eb_exc_amino_list:
                Eb_dict['Aminoglycosides'] = 1
        if df['AMIKACIN'] == 'I' or df['AMIKACIN'] == 'R':
            Eb_dict['Aminoglycosides'] = 1
        #抗MRSA头孢-头孢洛林
        #抗假单胞菌青霉素+β-内酰胺酶抑制剂 - 哌拉西林-他唑巴坦
        if df['PIPERACILLIN/TAZO'] == 'I' or df['PIPERACILLIN/TAZO'] == 'R':
            Eb_dict['AntiPenicillins&Lactamaseinhibitors'] = 1
        #碳青霉烯类 - 亚胺培南、美罗培南
        if (df[['IMIPENEM','MEROPENEM']] == 'I').any() or \
            (df[['IMIPENEM','MEROPENEM']] == 'R').any():
            Eb_dict['Carbapenems'] = 1
        #非超广谱头孢菌素-1、2代头孢 - 头孢唑林、头孢呋辛
        if df['CEFAZOLIN']=='I' or df['CEFAZOLIN']=='R':
            if df['org_name'] not in Eb_exc_cefazolin_list:
                Eb_dict['NESCephalosporins'] = 1
        if df['CEFUROXIME']=='I' or df['CEFUROXIME']=='R':
            if df['org_name'] not in Eb_exc_cefuroxime_list:
                Eb_dict['NESCephalosporins'] = 1
        #超广谱头孢菌素-3、4代头孢 - 头孢曲松、头孢他啶、头孢吡肟
        if (df[['CEFTRIAXONE','CEFTAZIDIME','CEFEPIME']]=='I').any() \
            or (df[['CEFTRIAXONE','CEFTAZIDIME','CEFEPIME']]=='R').any():
            Eb_dict['ESCephalosporins'] = 1
        #头孢菌素类 Cephamycins
        #氟喹诺酮类 - 环丙沙星
        if df['CIPROFLOXACIN'] == 'I' or df['CIPROFLOXACIN'] == 'R':
            Eb_dict['Fluoroquinolones'] = 1
        #叶酸途径抑制剂-甲氧苄啶 复方新诺明
        if df['TRIMETHOPRIM/SULFA']=='I' or df['TRIMETHOPRIM/SULFA']=='R':
            Eb_dict['Folate'] = 1
        #甘氨环素类 - 替加环素
        #单环β-内酰胺类 - 氨曲南
        #青霉素类 - 氨苄西林
        if df['AMPICILLIN']=='I' or df['AMPICILLIN']=='R':
            if df['org_name'] not in Eb_exc_ampicillin_list:
                Eb_dict['Penicillins'] = 1 
        #青霉素类+β内酰胺酶抑制剂 - 氨苄西林-舒巴坦
        if df['AMPICILLIN/SULBACTAM']=='I' or df['AMPICILLIN/SULBACTAM']=='R':
            if df['org_name'] not in Eb_exc_ampicillinsulbactam_list:
                Eb_dict['Penicillins&Lactamaseinhibitors'] = 1
        #糖肽类-万古霉素
        if df['VANCOMYCIN'] == 'I' or df['VANCOMYCIN'] == 'R':
            Eb_dict['Glycopeptides'] = 1
        #苯丙醇类
        #磷酶酸类
        #多粘菌素类
        #四环素类-四环素
        if df['TETRACYCLINE'] == 'I' or df['TETRACYCLINE'] == 'R':
            if df['org_name'] not in Eb_exc_teracycline_list:
                Eb_dict['Tetracyclines'] = 1
        
        for i in Eb_dict:
            Eb_sum += Eb_dict[i]
            Eb_ab_list.append(i)
        if Eb_sum>=3:
            return ";".join(Eb_ab_list), 1
        else:
            return ";".join(Eb_ab_list), 0
    #铜绿假单胞菌
    elif df['org_name'] in Ps_name:
        Ps_dict = {}
        Ps_sum = 0
        Ps_ab_list = []
        #氨基糖苷类 - 庆大霉素、妥布霉素、阿米卡星
        if (df[['GENTAMICIN','TOBRAMYCIN','AMIKACIN']] == 'I').any() \
            or (df[['GENTAMICIN','TOBRAMYCIN','AMIKACIN']] == 'R').any():
            Ps_dict['Aminoglycosides'] = 1    
        #抗假单胞菌碳青霉烯类-亚胺培南、美罗培南
        if (df[['IMIPENEM','MEROPENEM']] == 'I').any() or \
                (df[['IMIPENEM','MEROPENEM']] == 'R').any():
            Ps_dict['Carbapenems'] = 1
        #抗假单胞菌头孢菌素类-头孢他啶、头孢吡肟
        if (df[['CEFTAZIDIME','CEFEPIME']] == 'I').any() or \
                (df[['CEFTAZIDIME','CEFEPIME']] == 'R').any():
            Ps_dict['Cephalosporins'] = 1      
        #抗假单胞菌氟喹诺酮类-环丙沙星、左氧氟沙星
        if (df[['CIPROFLOXACIN','LEVOFLOXACIN']]=='I').any() or \
                (df[['CIPROFLOXACIN','LEVOFLOXACIN']]=='R').any():
            Ps_dict['Fluroquinolones'] = 1
        #抗假单胞菌青霉素类+β-内酰胺酶抑制剂
        if df['PIPERACILLIN/TAZO'] == 'I' or df['PIPERACILLIN/TAZO'] == 'R':
            Ps_dict['Penicillins&lactamaseinhibitors'] = 1
        #单环β-内酰胺类
        #磷霉酸类
        #多粘菌素类
        for i in Ps_dict:
            Ps_sum += Ps_dict[i]
            Ps_ab_list.append(i)
        if Ps_sum>=3:
            return ";".join(Ps_ab_list), 1
        else:
            return ";".join(Ps_ab_list), 0
    #不动杆菌属
    elif df['org_name'] in Ac_name:
        Ac_dict = {}
        Ac_sum = 0
        Ac_ab_list = []
        #氨基糖苷类 - 庆大霉素、妥布霉素、阿米卡星
        if (df[['GENTAMICIN','TOBRAMYCIN','AMIKACIN']] == 'I').any() \
            or (df[['GENTAMICIN','TOBRAMYCIN','AMIKACIN']] == 'R').any():
            Ac_dict['Aminoglycosides'] = 1    
        #抗假单胞菌碳青霉烯类 - 亚胺培南、美罗培南
        if (df[['IMIPENEM','MEROPENEM']] == 'I').any() or \
                (df[['IMIPENEM','MEROPENEM']] == 'R').any():
            Ac_dict['Carbapenems'] = 1
        #抗假单胞菌氟喹诺酮类 - 环丙沙星、左氧氟沙星
        if (df[['CIPROFLOXACIN','LEVOFLOXACIN']]=='I').any() or \
                (df[['CIPROFLOXACIN','LEVOFLOXACIN']]=='R').any():
            Ac_dict['Fluroquinolones'] = 1
        #抗假单胞菌青霉素类+β-内酰胺酶抑制剂
        if df['PIPERACILLIN/TAZO'] == 'I' or df['PIPERACILLIN/TAZO'] == 'R':
            Ac_dict['Penicillins&lactamaseinhibitors'] = 1
        #超广谱头孢菌素类-头孢他啶、头孢吡肟
        if (df[['CEFTAZIDIME','CEFEPIME']] == 'I').any() or \
                (df[['CEFTAZIDIME','CEFEPIME']] == 'R').any():
            Ac_dict['ESCephalosporins'] = 1
        #叶酸途径抑制剂-甲氧苄啶 复方新诺明
        if df['TRIMETHOPRIM/SULFA']=='I' or df['TRIMETHOPRIM/SULFA']=='R':
            Ac_dict['Folate'] = 1
        #青霉素类+β内酰胺酶抑制剂 - 氨苄西林-舒巴坦
        if df['AMPICILLIN/SULBACTAM']=='I' or df['AMPICILLIN/SULBACTAM']=='R':
            Ac_dict['Penicillins&lactamaseinhibitors'] = 1        
        #多粘菌素类
        #四环素类
        if df['TETRACYCLINE'] == 'I' or df['TETRACYCLINE'] == 'R':
            Ac_dict['Tetracyclines'] = 1        
        for i in Ac_dict:
            Ac_sum += Ac_dict[i]
            Ac_ab_list.append(i)
        if Ac_sum>=3:
            return ";".join(Ac_ab_list), 1
        else:
            return ";".join(Ac_ab_list), 0
    else:
        return np.nan, np.nan

In [26]:
MDR_CAL_cat[['resistant_ab_cat','MDR']] = MDR_CAL_cat.apply(MDR,axis = 1,result_type="expand")

In [27]:
MDR_CAL_cat['MDR'] = MDR_CAL_cat[['org_name','VANCOMYCIN','MDR']].\
apply(lambda x: 1 if ((x[0].find('ENTEROCOCCUS'))!=-1) \
and ((x[1] =='R') or (x[1] =='I')) \
else x[2],axis = 1)

MDR_CAL_cat = MDR_CAL_cat.drop_duplicates()

#查看每一类有多少耐药菌
for i,j in zip(['Staphylococcus aureus', 'Enterobacteriaceae', 'Enterococcus spp.', 
                'Pseudomonas aeruginosa','Acinetobacter spp.'],\
        [St_name, Eb_name, Ec_name, Ps_name, Ac_name]):
    print('-----------%s------------'%i)
    print(MDR_CAL_cat[MDR_CAL_cat.org_name.isin(j)].MDR.value_counts())

-----------Staphylococcus aureus------------
0.0    2899
1.0     990
Name: MDR, dtype: int64
-----------Enterobacteriaceae------------
0.0    2313
1.0     975
Name: MDR, dtype: int64
-----------Enterococcus spp.------------
0.0    1486
1.0     916
Name: MDR, dtype: int64
-----------Pseudomonas aeruginosa------------
0.0    735
1.0    216
Name: MDR, dtype: int64
-----------Acinetobacter spp.------------
0.0    71
1.0    55
Name: MDR, dtype: int64


In [28]:
len(MDR_CAL_cat.stay_id.unique())

7901

In [29]:
MDR_label = MDR_CAL_cat[['stay_id','micro_specimen_id', 'org_name','org_category', 'resistant_ab_cat', 'MDR']]

In [30]:
MDR_label = MDR_label.dropna(subset=['MDR'])

In [31]:
MDR_label.shape

(10656, 6)

In [32]:
#发生感染的
print('发生BI的：',BI.shape,len(pd.unique(BI.subject_id)),len(pd.unique(BI.hadm_id)),len(pd.unique(BI.stay_id)))
print('仅MDR检出结果：',len(MDR_label.stay_id.unique()),len(MDR_label.micro_specimen_id.unique()))

发生BI的： (87295, 20) 6332 6723 7901
仅MDR检出结果： 4906 7988


In [33]:
MDR_stay_id = MDR_label[['stay_id','org_category','MDR']]
MDR_stay_id = MDR_stay_id.sort_values(['stay_id','org_category','MDR'],ascending = False)
MDR_stay_id.org_category.value_counts()

Staphylococcus aureus     3889
Enterobacteriaceae        3288
Enterococcus spp.         2402
Pseudomonas aeruginosa     951
Acinetobacter spp.         126
Name: org_category, dtype: int64

In [34]:
MDR_stay_id = MDR_stay_id.drop_duplicates(subset = ['stay_id','org_category'],keep = 'first')
MDR_stay_id.shape

(5957, 3)

In [35]:
MDR_stay_id=MDR_stay_id.set_index(['stay_id', 'org_category'])['MDR'].unstack().reset_index()
MDR_stay_id['MDR_label'] = MDR_stay_id[
    ['Acinetobacter spp.', 'Enterobacteriaceae', 'Enterococcus spp.', 
     'Pseudomonas aeruginosa', 'Staphylococcus aureus']].eq(1).any(axis=1).astype(int)
MDR_stay_id.shape

(4906, 7)

In [36]:
MDR_stay_id.MDR_label.value_counts()

0    2941
1    1965
Name: MDR_label, dtype: int64

In [38]:
MDR_stay_id.to_csv('D:/2024BI_Journal/MIMIC_IV/Median_B/2_MDR_beforeICU_label.csv',index=False)

## 培养皿类型

In [46]:
SSC_ICU_IV = pd.read_csv('D:/2024BI_Journal/MIMIC_IV/1_SSC_ICU_IV.csv')
SSC_ICU_IV.shape

(531643, 20)

In [48]:
SSC_ICU_IV = SSC_ICU_IV[SSC_ICU_IV.stay_id.isin(pts.stay_id)]
SSC_ICU_IV.shape

(531643, 20)

In [49]:
SSC_ICU_IV['intime'] = pd.to_datetime(SSC_ICU_IV['intime'])
SSC_ICU_IV['outtime'] = pd.to_datetime(SSC_ICU_IV['outtime'])
SSC_ICU_IV['admittime'] = pd.to_datetime(SSC_ICU_IV['admittime'])

SSC_ICU_IV['charttime'] = pd.to_datetime(SSC_ICU_IV['charttime'])

# ICU
SSC_icu_window_criteria = SSC_ICU_IV['charttime'].between(SSC_ICU_IV['admittime'],SSC_ICU_IV['intime']+pd.DateOffset(1))

SSC_ICU_IV = SSC_ICU_IV[SSC_icu_window_criteria]

In [50]:
SSC_ICU_IV.shape

(237223, 20)

In [51]:
len(SSC_ICU_IV.stay_id.unique())

51762

In [52]:
# 判断规则
spec_types = {
    'BLOOD CULTURE': ['BLOOD CULTURE'],
    'URINE': ['URINE'],
    'SWAB': ['SWAB'],
    'SPUTUM': ['SPUTUM'],
    'MRSA SCREEN': ['MRSA SCREEN'],
    'BAL': ['BRONCHOALVEOLAR LAVAGE', 'BAL'],
    'TISSUE': ['TISSUE'],
    'PERITONEAL FLUID': ['PERITONEAL FLUID'],
    'PLEURAL FLUID': ['PLEURAL FLUID'],
    'ABSCESS': ['ABSCESS'],
    'CSF': ['CSF'],
    'JOINT FLUID': ['JOINT FLUID']
}

In [53]:
# 创建新的 DataFrame，初始为原始的 DataFrame
df_new = SSC_ICU_IV.copy()

# 为每个 spec_type_desc 创建新的列，初始值为0
for spec_type in spec_types:
    df_new[spec_type] = 0

# 遍历数据，检查每个 spec_type_desc 是否包含相关关键字，符合则更新对应列
for idx, row in df_new.iterrows():
    for spec_type, keywords in spec_types.items():
        # 检查关键词是否出现在 spec_type_desc 列的值中（不区分大小写）
        if any(keyword.lower() in row['spec_type_desc'].lower() for keyword in keywords):
            df_new.at[idx, spec_type] += 1

# 根据 stay_id 进行分组，聚合统计各列的值
df_new_grouped = df_new.groupby('stay_id').sum().reset_index()

In [54]:
df_new_grouped.columns

Index(['stay_id', 'microevent_id', 'micro_specimen_id', 'subject_id',
       'hadm_id', 'spec_itemid', 'org_itemid', 'los', 'BLOOD CULTURE', 'URINE',
       'SWAB', 'SPUTUM', 'MRSA SCREEN', 'BAL', 'TISSUE', 'PERITONEAL FLUID',
       'PLEURAL FLUID', 'ABSCESS', 'CSF', 'JOINT FLUID'],
      dtype='object')

In [55]:
df_new_grouped = df_new_grouped[['stay_id', 'BLOOD CULTURE', 'URINE', 'SWAB',
       'SPUTUM', 'MRSA SCREEN', 'BAL', 'TISSUE', 'PERITONEAL FLUID',
       'PLEURAL FLUID', 'ABSCESS', 'CSF', 'JOINT FLUID']]
df_new_grouped.columns = ['ICUSTAY_ID', 'BLOOD CULTURE', 'URINE', 'SWAB', 'SPUTUM', 'MRSA SCREEN',
       'BAL', 'TISSUE', 'PERITONEAL FLUID', 'PLEURAL FLUID', 'ABSCESS', 'CSF',
       'JOINT FLUID']

In [57]:
df_new_grouped.head(2)

,ICUSTAY_ID,BLOOD CULTURE,URINE,SWAB,SPUTUM,MRSA SCREEN,BAL,TISSUE,PERITONEAL FLUID,PLEURAL FLUID,ABSCESS,CSF,JOINT FLUID
0,30000153,0,0,0,0,1,0,0,0,0,0,0,0
1,30000484,2,2,0,0,1,0,0,0,0,0,0,0


In [58]:
len(df_new_grouped.ICUSTAY_ID.unique())

51762

In [59]:
df_new_grouped.to_csv('D:/2024BI_Journal/MIMIC_IV/Median_B/All_culture_type.csv',index=False)